In [2]:
import pandas as pd

movies = pd.read_csv("movies.csv")
ratings = pd.read_csv("ratings.csv")
tags = pd.read_csv("tags.csv")

# aggregate tags into one string per movie (many users tag the same movie differently)
tags_agg = tags.groupby("movieId")["tag"].apply(lambda t: "|".join(t.dropna().unique())).reset_index()
tags_agg.columns = ["movieId", "tags"]

# rating stats per movie
rating_stats = ratings.groupby("movieId")["rating"].agg(["count", "mean"]).reset_index()
rating_stats.columns = ["movieId", "rating_count", "rating_mean"]

full = movies.merge(rating_stats, on="movieId", how="left").merge(tags_agg, on="movieId", how="left")
full.head()


,movieId,title,genres,rating_count,rating_mean,tags
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,215.0,3.920930,pixar|fun
1,2,Jumanji (1995),Adventure|Children|Fantasy,110.0,3.431818,fantasy|magic board game|Robin Williams|game
2,3,Grumpier Old Men (1995),Comedy|Romance,52.0,3.259615,moldy|old
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,7.0,2.357143,NaN
4,5,Father of the Bride Part II (1995),Comedy,49.0,3.071429,pregnancy|remake


In [10]:
cf_ratings = ratings[ratings["movieId"].isin(full["movieId"])][["userId", "movieId", "rating", "timestamp"]]

print(f"ratings before filtering: {len(ratings)}")
print(f"ratings after filtering to cleaned movie list: {len(cf_ratings)}")
print(f"unique users: {cf_ratings['userId'].nunique()}, unique movies: {cf_ratings['movieId'].nunique()}")

cf_ratings = cf_ratings.drop(columns=['timestamp'])


ratings before filtering: 100836
ratings after filtering to cleaned movie list: 100836
unique users: 610, unique movies: 9724


In [11]:
cf_ratings.head()

,userId,movieId,rating
0,1,1,4.0
1,1,3,4.0
2,1,6,4.0
3,1,47,5.0
4,1,50,5.0


In [13]:
full = full.drop(columns=['rating_count'])
full = full.drop(columns=['rating_mean'])


KeyError: "['rating_count'] not found in axis"

In [14]:
full.head()

,movieId,title,genres,tags
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,pixar|fun
1,2,Jumanji (1995),Adventure|Children|Fantasy,fantasy|magic board game|Robin Williams|game
2,3,Grumpier Old Men (1995),Comedy|Romance,moldy|old
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,NaN
4,5,Father of the Bride Part II (1995),Comedy,pregnancy|remake


In [16]:
import json
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

# 1. Prepare text features from content dataframe
content_df = text_data.copy()  # Replace 'text_data' with your content dataframe name
content_df["tags"] = content_df["tags"].fillna("")
content_df["genres_clean"] = content_df["genres"].str.replace("|", " ", regex=False)
content_df["tags_clean"] = content_df["tags"].str.replace("|", " ", regex=False)

# Combine title, genres, and tags into a single text representation
content_df["combined_text"] = (
    content_df["title"] + " " + 
    content_df["genres_clean"] + " " + 
    content_df["tags_clean"]
)

# 2. Vectorize texts
model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = model.encode(
    content_df["combined_text"].tolist(), 
    show_progress_bar=True, 
    convert_to_numpy=True
)

# 3. Build FAISS index for fast similarity search
faiss.normalize_L2(embeddings)
index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)

# 4. Save artifacts directly into current working directory (inside data/)
np.save("content_embeddings.npy", embeddings)
faiss.write_index(index, "faiss.index")
with open("movie_ids.json", "w") as f:
    json.dump(content_df["movieId"].tolist(), f)

print(f"Content model ready! Embedded {len(content_df)} movies (dim={embeddings.shape[1]}).")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/305 [00:00<?, ?it/s]

Content model ready! Embedded 9745 movies (dim=384).


In [17]:
import pickle
from surprise import SVD, Dataset, Reader

# 1. Define rating scale for Surprise (typically 0.5 to 5.0 for MovieLens)
reader = Reader(rating_scale=(0.5, 5.0))

# 2. Load dataset from your CF dataframe
# Replace 'cf_ratings' with your CF dataframe variable name
data = Dataset.load_from_df(cf_ratings[["userId", "movieId", "rating"]], reader)
trainset = data.build_full_trainset()

# 3. Train SVD
cf_model = SVD(n_factors=100, n_epochs=20, random_state=42)
cf_model.fit(trainset)

# 4. Save model artifact directly into current working directory (inside data/)
with open("cf_model.pkl", "wb") as f:
    pickle.dump(cf_model, f)

print(f"CF SVD model successfully trained on {trainset.n_ratings} ratings and saved to cf_model.pkl!")

CF SVD model successfully trained on 100836 ratings and saved to cf_model.pkl!
